# Mining Dispatch — Ejecutar desde GitHub en Colab

> **Ventaja:** Clona el repositorio completo con todo el código, configuración y scripts
> **Requisito:** Tu repositorio debe estar en GitHub con el modelo y videos

## Paso 1: Verificar GPU

In [ ]:
!nvidia-smi
import torch
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## Paso 2: Clonar repositorio desde GitHub

In [ ]:
# Cambia esto por tu repositorio real
GITHUB_REPO = 'https://github.com/AcerosDavid/IA-Dis.git'

!git clone {GITHUB_REPO}

import os
os.chdir('/content/IA-Dis')
print('Repositorio clonado. Directorio actual:', os.getcwd())

## Paso 3: Verificar estructura del proyecto

In [ ]:
!ls -la
print('\nEstructura del proyecto:')
!find . -type f -name '*.py' | head -20

## Paso 4: Instalar dependencias

In [ ]:
!pip install -r requirements.txt -q

# Instalar dependencias opcionales para Colab
!pip install easyocr onnxruntime onnx onnxsim -q

import ultralytics, torch, cv2
print('ultralytics:', ultralytics.__version__)
print('torch:', torch.__version__)
print('opencv:', cv2.__version__)
print('CUDA:', torch.cuda.is_available())

## Paso 5: Verificar archivos necesarios

In [ ]:
from pathlib import Path

# Verificar modelo
model_path = Path('models/best.pt')
if model_path.exists():
    print(f'✓ Modelo encontrado: {model_path}')
else:
    print(f'✗ Modelo NO encontrado: {model_path}')
    print('Necesitas subir best.pt al repositorio o montar Drive')

# Verificar videos
video_dir = Path('data/raw')
if video_dir.exists():
    videos = list(video_dir.glob('*.mp4')) + list(video_dir.glob('*.avi'))
    if videos:
        print(f'✓ Videos encontrados: {len(videos)}')
        for v in videos[:5]:
            print(f'  - {v.name}')
    else:
        print('✗ No hay videos en data/raw/')
else:
    print('✗ Directorio data/raw/ no existe')

# Verificar config del tracker
tracker_cfg = Path('cfg/botsort_mining.yaml')
if tracker_cfg.exists():
    print(f'✓ Config tracker: {tracker_cfg}')
else:
    print(f'✗ Config tracker NO encontrado: {tracker_cfg}')

## Paso 6: Montar Drive (opcional - para modelos/videos grandes)

In [ ]:
# Solo necesitas esto si el modelo/video no están en el repo
from google.colab import drive
drive.mount('/content/drive')

# Copiar modelo desde Drive si no está en el repo
from pathlib import Path
import shutil

drive_model = Path('/content/drive/MyDrive/mining_dispatch/best.pt')
local_model = Path('models/best.pt')

if not local_model.exists() and drive_model.exists():
    print('Copiando modelo desde Drive...')
    shutil.copy(drive_model, local_model)
    print(f'Modelo copiado: {local_model}')
elif local_model.exists():
    print('Modelo ya existe en el repo')
else:
    print('ERROR: Modelo no encontrado ni en repo ni en Drive')

## Paso 7: Ejecutar detección usando el script del proyecto

In [ ]:
# Usar el script runner del proyecto
import sys
sys.path.insert(0, '/content/IA-Dis')

from pathlib import Path

# Configurar parámetros
VIDEO_PATH = 'data/raw/video.mp4'  # Cambia esto por tu video
MODEL_PATH = 'models/best.pt'
OUTPUT_PATH = 'data/output/video_anotado.mp4'

# Crear directorio de output
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

# Verificar que existen
if not Path(VIDEO_PATH).exists():
    print(f'ERROR: Video no encontrado: {VIDEO_PATH}')
    print('Videos disponibles:')
    for v in Path('data/raw').glob('*.mp4'):
        print(f'  - {v}')
elif not Path(MODEL_PATH).exists():
    print(f'ERROR: Modelo no encontrado: {MODEL_PATH}')
else:
    print(f'Video: {VIDEO_PATH}')
    print(f'Modelo: {MODEL_PATH}')
    print(f'Output: {OUTPUT_PATH}')
    print('\nEjecutando detección...')
    
    # Ejecutar usando el script del proyecto
    !python colab_runner.py \
        --video {VIDEO_PATH} \
        --model {MODEL_PATH} \
        --output {OUTPUT_PATH} \
        --imgsz 640 \
        --confidence 0.5

## Paso 8: O ejecutar usando el pipeline completo del proyecto

In [ ]:
# Alternativa: Usar el pipeline completo del proyecto (sin UI)
# Esto usa src/pipeline/runner.py y todas las optimizaciones

import sys
sys.path.insert(0, '/content/IA-Dis')

from pathlib import Path
import os

# Configurar variables de entorno
os.environ['MODEL_PATH'] = 'models/best.pt'
os.environ['DB_URL'] = 'sqlite:///data/eventos_colab.db'
os.environ['VIDEO_PATH'] = 'data/raw/video.mp4'
os.environ['CONFIDENCE_THRESHOLD'] = '0.5'
os.environ['TRACK_SKIP'] = '1'  # En GPU podemos procesar todos los frames
os.environ['TRACK_IMGSZ'] = '640'
os.environ['OCR_HABILITADO'] = 'false'
os.environ['CLASIFICADOR_HABILITADO'] = 'false'

# Crear archivo .env temporal
env_content = '''MODEL_PATH=models/best.pt
DB_URL=sqlite:///data/eventos_colab.db
VIDEO_PATH=data/raw/video.mp4
CONFIDENCE_THRESHOLD=0.5
TRACK_SKIP=1
TRACK_IMGSZ=640
OCR_HABILITADO=false
CLASIFICADOR_HABILITADO=false
'''

Path('.env').write_text(env_content)
print('.env creado')

# Ejecutar un script simplificado que use el pipeline completo
# Nota: app.py no funcionará en Colab porque usa Tkinter
# Necesitamos crear un script sin UI

print('Para usar el pipeline completo, necesitaríamos crear un script sin UI.')
print('Por ahora, usa colab_runner.py que está optimizado para Colab.')

## Paso 9: Descargar resultados

In [ ]:
from google.colab import files
from pathlib import Path

output_path = Path('data/output/video_anotado.mp4')

if output_path.exists():
    print(f'Descargando video anotado...')
    files.download(str(output_path))
else:
    print('Video anotado no encontrado')
    print('Archivos en data/output/:')
    if Path('data/output').exists():
        for f in Path('data/output').iterdir():
            print(f'  - {f.name}')

## Paso 10: Copiar resultados a Drive (opcional)

In [ ]:
import shutil
from pathlib import Path
from datetime import datetime

# Crear directorio en Drive para resultados
drive_output = Path('/content/drive/MyDrive/mining_dispatch/colab_results')
drive_output.mkdir(parents=True, exist_ok=True)

# Copiar video anotado
output_path = Path('data/output/video_anotado.mp4')
if output_path.exists():
    timestamp = datetime.now().strftime('%Y%m%d_%H%M')
    dest = drive_output / f'video_anotado_{timestamp}.mp4'
    shutil.copy(output_path, dest)
    print(f'Video copiado a Drive: {dest}')

# Copiar base de datos si existe
db_path = Path('data/eventos_colab.db')
if db_path.exists():
    shutil.copy(db_path, drive_output / 'eventos_colab.db')
    print(f'BD copiada a Drive: {drive_output / "eventos_colab.db"}')

print(f'\nTodos los resultados guardados en: {drive_output}')

## Resumen de ventajas de clonar desde GitHub

### ✅ Ventajas:
- Todo el código disponible (src/, scripts/, tests/)
- Configuración completa (cfg/, .env.example)
- Scripts de entrenamiento y datasets disponibles
- Fácil actualizar con `git pull`
- Version control integrado

### ⚠️ Limitaciones:
- El modelo `.pt` no debería estar en el repo (muy grande)
- Los videos no deberían estar en el repo (muy grandes)
- Solución: Usar Drive para modelos/videos grandes

### 📁 Estructura recomendada del repo:
```
IA-Dis/
├── src/              # Todo el código ✓
├── cfg/              # Configuración ✓
├── scripts/          # Scripts de entrenamiento ✓
├── tests/            # Tests ✓
├── .env.example      # Plantilla de configuración ✓
├── requirements.txt  # Dependencias ✓
├── colab_runner.py   # Script para Colab ✓
├── models/           # .gitkeep (sin .pt)
└── data/             # .gitkeep (sin videos)
```

### 🔄 Flujo de trabajo:
1. Clonar repo en Colab
2. Montar Drive para modelo/videos grandes
3. Copiar modelo/videos desde Drive al proyecto
4. Ejecutar detección
5. Guardar resultados en Drive
6. (Opcional) Hacer commit de mejoras al repo